# 10 — Multi-Modal Infrastructure

## Background

Modern LLM applications rarely process text alone. Users upload images, PDFs, voice messages, and spreadsheets. Production infrastructure must handle diverse modalities without turning every endpoint into a spaghetti of special cases.

## What You'll Learn

- How to build a unified multi-modal pipeline with typed routing
- Image preprocessing: base64 encoding, resizing, format normalization
- Audio handling: chunking, transcription stub, speaker-diarisation metadata
- Document parsing: PDF page extraction, table detection, chunked embedding
- Composing modality-specific processors behind a single `ModalityRouter`
- Cost and latency estimation per modality

## Why This Matters

Multi-modal handling is where infrastructure complexity explodes. The same patterns that work for text (streaming, caching, retries) need modality-aware adaptations. Getting the abstraction right early prevents rewriting every consumer when you add a new modality.

In [ ]:
import base64, io, json, time, hashlib, mimetypes
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Tuple
from enum import Enum
import numpy as np

class Modality(str, Enum):
    TEXT     = 'text'
    IMAGE    = 'image'
    AUDIO    = 'audio'
    DOCUMENT = 'document'
    UNKNOWN  = 'unknown'

MIME_TO_MODALITY = {
    'text/plain': Modality.TEXT,
    'text/html':  Modality.TEXT,
    'image/jpeg': Modality.IMAGE,
    'image/png':  Modality.IMAGE,
    'image/webp': Modality.IMAGE,
    'audio/mpeg': Modality.AUDIO,
    'audio/wav':  Modality.AUDIO,
    'audio/ogg':  Modality.AUDIO,
    'application/pdf': Modality.DOCUMENT,
}

def detect_modality(mime_type: str) -> Modality:
    return MIME_TO_MODALITY.get(mime_type.split(';')[0].strip(), Modality.UNKNOWN)

print('Modality detection:')
for mime in ['image/png', 'audio/wav', 'application/pdf', 'text/plain', 'video/mp4']:
    print(f'  {mime:30s} -> {detect_modality(mime).value}')


## Typed Asset Representation

Every modality enters the pipeline as a `ModalAsset` — a container that tracks the raw bytes, metadata, and preprocessing results side-by-side.

In [ ]:
@dataclass
class ModalAsset:
    asset_id:   str
    modality:   Modality
    mime_type:  str
    raw_bytes:  bytes
    metadata:   Dict[str, Any] = field(default_factory=dict)
    processed:  Optional[Dict[str, Any]] = None
    cost_units: float = 0.0

    @classmethod
    def from_bytes(cls, data: bytes, mime_type: str) -> 'ModalAsset':
        asset_id = hashlib.sha256(data).hexdigest()[:16]
        return cls(
            asset_id  = asset_id,
            modality  = detect_modality(mime_type),
            mime_type = mime_type,
            raw_bytes = data,
            metadata  = {'size_bytes': len(data), 'mime': mime_type},
        )

    def __repr__(self):
        return (f'ModalAsset(id={self.asset_id}, modality={self.modality.value}, '
                f'size={len(self.raw_bytes)}B, processed={self.processed is not None})')

fake_image_bytes = b'\x89PNG\r\n' + bytes(range(256)) * 4
fake_audio_bytes = b'RIFF' + bytes(range(128)) * 8
fake_doc_bytes   = b'%PDF-1.4\n' + bytes(range(64)) * 16

assets = [
    ModalAsset.from_bytes(fake_image_bytes, 'image/png'),
    ModalAsset.from_bytes(fake_audio_bytes, 'audio/wav'),
    ModalAsset.from_bytes(fake_doc_bytes,   'application/pdf'),
]
for a in assets:
    print(a)


## Modality-Specific Processors

In [ ]:
from abc import ABC, abstractmethod

class ModalityProcessor(ABC):
    @abstractmethod
    def process(self, asset: ModalAsset) -> ModalAsset: ...

class ImageProcessor(ModalityProcessor):
    def __init__(self, max_side: int = 1024):
        self.max_side = max_side

    def process(self, asset: ModalAsset) -> ModalAsset:
        t0 = time.perf_counter()
        w, h = 1920, 1080
        scale = min(self.max_side / w, self.max_side / h)
        new_w, new_h = int(w * scale), int(h * scale)
        resized = asset.raw_bytes[:len(asset.raw_bytes) // 2]
        b64 = base64.b64encode(resized).decode()
        asset.processed = {
            'type':       'image_url',
            'image_url':  {'url': f'data:{asset.mime_type};base64,{b64[:64]}...'},
            'dimensions': {'width': new_w, 'height': new_h},
            'latency_ms': round((time.perf_counter() - t0) * 1000, 2),
        }
        asset.cost_units = (new_w * new_h) / (512 * 512)
        return asset

class AudioProcessor(ModalityProcessor):
    def __init__(self, chunk_duration_s: float = 30.0, sample_rate: int = 16000):
        self.chunk_duration_s = chunk_duration_s
        self.sample_rate      = sample_rate

    def process(self, asset: ModalAsset) -> ModalAsset:
        t0       = time.perf_counter()
        duration = len(asset.raw_bytes) / (self.sample_rate * 2)
        chunk_bytes = int(self.chunk_duration_s * self.sample_rate * 2)
        chunks   = [asset.raw_bytes[i:i+chunk_bytes]
                    for i in range(0, len(asset.raw_bytes), chunk_bytes)]
        asset.processed = {
            'type':       'audio_transcript',
            'transcript': f'[TRANSCRIPT_STUB: {len(chunks)} chunks, {duration:.1f}s]',
            'duration_s': round(duration, 2),
            'n_chunks':   len(chunks),
            'latency_ms': round((time.perf_counter() - t0) * 1000, 2),
        }
        asset.cost_units = duration / 60.0
        return asset

class DocumentProcessor(ModalityProcessor):
    def __init__(self, chunk_size: int = 512, overlap: int = 64):
        self.chunk_size = chunk_size
        self.overlap    = overlap

    def process(self, asset: ModalAsset) -> ModalAsset:
        t0     = time.perf_counter()
        n_pages = max(1, len(asset.raw_bytes) // 1000)
        pages  = [f'Page {i+1}: lorem ipsum paragraph {i+1}...' for i in range(n_pages)]
        full   = ' '.join(pages)
        words  = full.split()
        step   = self.chunk_size - self.overlap
        chunks = [' '.join(words[i:i+self.chunk_size])
                  for i in range(0, max(1, len(words) - self.overlap), step)]
        asset.processed = {
            'type':       'document_chunks',
            'n_pages':    n_pages,
            'n_chunks':   len(chunks),
            'total_chars': len(full),
            'latency_ms': round((time.perf_counter() - t0) * 1000, 2),
        }
        asset.cost_units = n_pages
        return asset

procs = {Modality.IMAGE: ImageProcessor(), Modality.AUDIO: AudioProcessor(), Modality.DOCUMENT: DocumentProcessor()}
for a in assets:
    proc = procs.get(a.modality)
    if proc:
        proc.process(a)
        print(f'{a.modality.value.upper()}: {a.processed}')


## Modality Router

The router is the single entry point. It detects mime types, selects the right processor, and builds a unified LLM payload.

In [ ]:
class ModalityRouter:
    def __init__(self):
        self.processors: Dict[Modality, ModalityProcessor] = {
            Modality.IMAGE:    ImageProcessor(),
            Modality.AUDIO:    AudioProcessor(),
            Modality.DOCUMENT: DocumentProcessor(),
        }

    def route(self, data: bytes, mime_type: str) -> ModalAsset:
        asset = ModalAsset.from_bytes(data, mime_type)
        proc  = self.processors.get(asset.modality)
        if proc is None:
            raise ValueError(f'No processor for modality: {asset.modality}')
        return proc.process(asset)

    def build_llm_payload(self, text_prompt: str, assets: List[ModalAsset]) -> Dict:
        content = [{'type': 'text', 'text': text_prompt}]
        for a in assets:
            if a.processed:
                content.append(a.processed)
        return {
            'model':    'claude-sonnet-4-6',
            'max_tokens': 1024,
            'messages': [{'role': 'user', 'content': content}],
        }

    def cost_report(self, assets: List[ModalAsset]) -> Dict:
        by_mod = {}
        for a in assets:
            by_mod[a.modality.value] = by_mod.get(a.modality.value, 0) + a.cost_units
        return {'total_units': sum(by_mod.values()), 'by_modality': by_mod}

router  = ModalityRouter()
routed  = [router.route(a.raw_bytes, a.mime_type) for a in assets]
payload = router.build_llm_payload('Describe everything in these files.', routed)
report  = router.cost_report(routed)

print('LLM Payload:')
print(f'  model: {payload["model"]}')
print(f'  content blocks: {len(payload["messages"][0]["content"])}')
for block in payload['messages'][0]['content']:
    print(f'    type={block["type"]}')
print(f'Cost report: {report}')


## Latency Benchmark

In [ ]:
import random
rng = np.random.default_rng(42)
test_cases = [
    (fake_image_bytes, 'image/png'),
    (fake_audio_bytes, 'audio/wav'),
    (fake_doc_bytes,   'application/pdf'),
] * 20
random.shuffle(test_cases)

router2   = ModalityRouter()
latencies: Dict[str, List[float]] = {}

for data, mime in test_cases:
    t0    = time.perf_counter()
    asset = router2.route(data, mime)
    ms    = (time.perf_counter() - t0) * 1000
    key   = asset.modality.value
    latencies.setdefault(key, []).append(ms)

print(f'  {"Modality":<12} {"Count":>6} {"Mean":>8} {"p50":>8} {"p95":>8}')
for mod, lats in sorted(latencies.items()):
    arr = np.array(lats)
    print(f'  {mod:<12} {len(arr):>6} {arr.mean():>8.3f} '
          f'{np.percentile(arr,50):>8.3f} {np.percentile(arr,95):>8.3f}')


## Key Takeaways

- `ModalAsset` is the typed container — processors write to `processed`, consumers read from it
- Processors are registered by modality; adding video or CSV requires only a new processor class
- `cost_units` are modality-specific (image tiles, audio minutes, PDF pages) and accumulate at the router level
- The payload builder composes multi-part LLM messages without callers needing to know encoding details